# DiffuSVG — Complete 5-Stage Pipeline

**Prompt → Diffusion → Vectorize → VLM Gate → Fine-tune → Correction → Eval**

| Stage | What it does | Saved outputs |
|-------|--------------|---------------|
| 1 | SD 3.5 Medium + Potrace → SVG training pairs | PNGs, SVGs, JSONL |
| 2 | VLM Quality Gate (Qwen2-VL-2B YES/NO) | Kept/rejected renders, JSONL |
| 3 | QLoRA Fine-Tuning (Qwen2-VL-2B + LoRA) | Training curve, adapter zip |
| 4 | Inference + Code Correction (text → SVG) | SVG files, rendered PNGs, JSON |
| 5 | Evaluation: CLIP + model comparison | SVGs, renders, plots, results JSON |

> **Before running**: Runtime → Change runtime type → **GPU** (T4 minimum, A100 recommended)
>
> **Run order**: Cell 1 (Install) → **Restart Runtime** → Cell 2 onwards


In [ ]:
#@title Install Dependencies (run ONCE, then restart runtime)
# transformers>=4.47 + diffusers>=0.32 must be installed together.
# diffusers<=0.31 imports FLAX_WEIGHTS_NAME which was removed in transformers>=4.47.
!pip uninstall -y transformers accelerate bitsandbytes diffusers huggingface_hub -q 2>/dev/null
!pip install 'huggingface_hub>=0.26.2' -U -q
!pip install 'transformers>=4.47.0' 'accelerate>=0.35.0' -q
!pip install bitsandbytes 'diffusers>=0.32.0' peft trl -q
!pip install qwen-vl-utils cairosvg open_clip_torch -q
!pip install pillow torchvision numpy pandas matplotlib tqdm scikit-learn -q
!apt-get install -qq potrace imagemagick libcairo2-dev
print('=' * 50)
print('Done!  Runtime > Restart Runtime')
print('Then run Config cell, skip this one.')
print('=' * 50)


In [ ]:
#@title Configuration { display-mode: "form" }
import os

HF_TOKEN = ''  #@param {type:"string"}

# Load HF token from Kaggle Secrets
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

# Toggle which stages to run
RUN_STAGE1 = True  #@param {type:"boolean"}
RUN_STAGE2 = True  #@param {type:"boolean"}
RUN_STAGE3 = True  #@param {type:"boolean"}
RUN_STAGE4 = True  #@param {type:"boolean"}
RUN_STAGE5 = True  #@param {type:"boolean"}

# Stage 1 — Data generation
NUM_GEN_PROMPTS = 50   #@param {type:"slider", min:10, max:200, step:10}
DIFF_STEPS      = 30   #@param {type:"slider", min:20, max:50, step:5}
GUIDANCE        = 5.0  #@param {type:"slider", min:3.0, max:10.0, step:0.5}
N_COLORS        = 6    #@param {type:"slider", min:2, max:12, step:1}

# Stage 3 — Fine-tuning
BASE_MODEL = 'Qwen/Qwen2-VL-2B-Instruct'  #@param ["Qwen/Qwen2-VL-2B-Instruct", "Qwen/Qwen2-VL-7B-Instruct"]
LORA_R     = 32    #@param {type:"slider", min:8, max:64, step:8}
LORA_ALPHA = 64    #@param {type:"slider", min:16, max:128, step:16}
EPOCHS     = 5     #@param {type:"slider", min:1, max:10, step:1}
LR         = 1e-4  #@param {type:"number"}
MAX_SEQ    = 2048  #@param {type:"slider", min:1024, max:4096, step:512}

# Stage 4 — Inference
ADAPTER_PATH    = './qwen2vl_svg_lora/final_adapter'  #@param {type:"string"}
MAX_CORR_ROUNDS = 3     #@param {type:"slider", min:0, max:5, step:1}
MAX_NEW_TOKENS  = 1500  #@param {type:"slider", min:512, max:3000, step:256}

# Stage 5 — API baselines (optional)
OPENAI_API_KEY    = ''  #@param {type:"string"}
ANTHROPIC_API_KEY = ''  #@param {type:"string"}

# ── Output folder structure ────────────────────────────────────────────────────
DATASET_DIR    = './dataset'
OUTPUT_DIR     = './qwen2vl_svg_lora'
RESULTS_FILE   = './results.json'
EVAL_PROMPTS   = './eval_prompts.json'

S1_IMAGES  = './outputs/stage1/images'   # SD-generated PNGs
S1_SVGS    = './outputs/stage1/svgs'     # vectorized SVGs
S2_KEPT    = './outputs/stage2/kept'     # rendered PNGs of kept pairs
S2_REJECT  = './outputs/stage2/rejected' # rendered PNGs of rejected pairs
S3_OUT     = './outputs/stage3'          # training curve + adapter zip
S4_SVGS    = './outputs/stage4/svgs'     # generated SVG files
S4_RENDERS = './outputs/stage4/renders'  # rendered PNGs
S5_SVGS    = './outputs/stage5/svgs'     # eval SVG files
S5_RENDERS = './outputs/stage5/renders'  # eval rendered PNGs

for d in [DATASET_DIR, OUTPUT_DIR,
          S1_IMAGES, S1_SVGS,
          S2_KEPT, S2_REJECT,
          S3_OUT,
          S4_SVGS, S4_RENDERS,
          S5_SVGS, S5_RENDERS]:
    os.makedirs(d, exist_ok=True)

if HF_TOKEN and len(HF_TOKEN) > 20:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print('HuggingFace authenticated')
else:
    print('HF_TOKEN not set — SD 3.5 Medium requires it for Stage 1')
print('Config ready. Output folders created under ./outputs/')


In [ ]:
#@title Core Imports + SVG Utilities
import gc, io, json, os, re, subprocess, tempfile, time, warnings, zipfile, random, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn.functional as F
import cairosvg
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

if not torch.cuda.is_available():
    raise RuntimeError('No GPU detected. On Kaggle: Settings > Accelerator > GPU T4 x2 or P100')
DEVICE = 'cuda'
print(f'GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)')

SVG_SYSTEM_PROMPT = (
    'You are an expert SVG code generator. '
    'Given a text description, output ONLY valid SVG code.\n'
    'Rules:\n'
    '- Start with: <svg viewBox="0 0 200 200" xmlns="http://www.w3.org/2000/svg">\n'
    '- Use simple shapes: <rect>, <circle>, <ellipse>, <polygon>, <path>\n'
    '- Use solid hex fill colors (e.g., fill="#FF0000")\n'
    '- Keep it minimal: 5-20 elements\n'
    '- End with: </svg>\n'
    'Output SVG code directly, no explanation.'
)

def slug(text, maxlen=40):
    return re.sub(r'\W+', '_', text)[:maxlen].strip('_')

def extract_svg(text):
    if not text: return None
    if '```' in text:
        for part in text.split('```'):
            p = part.strip().lstrip('svg').lstrip('xml').strip()
            if p.startswith('<svg'): text = p; break
    m = re.search(r'<svg[\s>]', text)
    if m: text = text[m.start():]
    end = text.rfind('</svg>')
    if end != -1: text = text[:end + 6]
    return text.strip() if '<svg' in text else None

def repair_svg(svg):
    if not svg: return '<svg viewBox="0 0 200 200" xmlns="http://www.w3.org/2000/svg"></svg>'
    svg = svg.strip()
    m = re.search(r'<svg[\s>]', svg)
    if m: svg = svg[m.start():]
    svg += '</g>' * max(0, len(re.findall(r'<g\b[^>]*>', svg)) - len(re.findall(r'</g>', svg)))
    if not svg.rstrip().endswith('</svg>'):
        svg = svg.rstrip() + ('\n</svg>' if svg.endswith('>') else '" fill="#000"/>\n</svg>')
    return svg

def validate_svg(svg):
    if not svg or '<svg' not in svg: return False
    try:
        cairosvg.svg2png(bytestring=svg.encode(), output_width=64, output_height=64)
        return True
    except: return False

def render_svg(svg, size=224):
    try:
        png = cairosvg.svg2png(bytestring=svg.encode(), output_width=size, output_height=size)
        return Image.open(io.BytesIO(png)).convert('RGB')
    except: return None

def count_elements(svg):
    return len(re.findall(r'<(path|rect|circle|ellipse|polygon|polyline|line)\b', svg or '', re.I))

def download_zip(zip_path, label=''):
    """Download zip — Colab only. On Kaggle, find it in the Output panel."""
    on_kaggle = os.path.exists('/kaggle')
    if on_kaggle:
        print(f'Output saved: {zip_path} {label}')
        print('  -> On Kaggle: open the Output panel (right sidebar) to download.')
        return
    try:
        from google.colab import files
        files.download(zip_path)
        print(f'Downloading {zip_path} {label}')
    except Exception as e:
        print(f'Saved locally: {zip_path} {label}')

def zip_folder(folder, zip_path):
    """Zip an entire folder."""
    folder = Path(folder)
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in sorted(folder.rglob('*')):
            if f.is_file():
                zf.write(f, f.relative_to(folder.parent))
    size_mb = Path(zip_path).stat().st_size / 1e6
    count   = sum(1 for _ in folder.rglob('*') if _.is_file())
    print(f'  Zipped {count} files -> {zip_path} ({size_mb:.1f} MB)')

def save_results(results, path=RESULTS_FILE):
    valid = [r for r in results if r.get('success')]
    clips = [r['clip'] for r in valid if r.get('clip', 0) > 0]
    summary = {
        'n_total':   len(results),
        'n_valid':   len(valid),
        'valid_pct': round(100 * len(valid) / max(len(results), 1), 1),
        'clip_mean': round(float(np.mean(clips)), 4) if clips else 0.0,
        'clip_std':  round(float(np.std(clips)),  4) if clips else 0.0,
    }
    export = [{k: v for k, v in r.items() if not k.startswith('_')} for r in results]
    with open(path, 'w') as f:
        json.dump({'summary': summary, 'results': export}, f, indent=2)
    pd.DataFrame(export).to_csv(str(path).replace('.json', '.csv'), index=False)
    return summary

print('SVG utilities ready')


---
## Stage 1 — Data Generation
**SD 3.5 Medium → raster PNG (512×512) → k-means color quantization → Potrace per color → SVG**

Saves: `outputs/stage1/images/*.png`, `outputs/stage1/svgs/*.svg`, `dataset/raw_dataset.jsonl`


In [ ]:
#@title Stage 1a — Load SD 3.5 Medium
if RUN_STAGE1:
    from diffusers import StableDiffusion3Pipeline
    print('Loading SD 3.5 Medium (~3 min first time)...')
    sd_pipe = StableDiffusion3Pipeline.from_pretrained(
        'stabilityai/stable-diffusion-3.5-medium',
        torch_dtype=torch.float16,
        token=HF_TOKEN
    )
    sd_pipe.enable_model_cpu_offload()
    NEG   = 'gradient, shadow, texture, 3d, realistic, photograph, complex details, noise'
    STYLE = 'ultra-simple flat vector, geometric shapes only, solid colors, icon style, minimalist, '

    @torch.inference_mode()
    def generate_image(prompt, seed=42):
        gen = torch.Generator('cuda').manual_seed(seed)
        return sd_pipe(
            STYLE + prompt, negative_prompt=NEG,
            num_inference_steps=DIFF_STEPS, guidance_scale=GUIDANCE,
            generator=gen
        ).images[0]

    print(f'SD 3.5 Medium ready (steps={DIFF_STEPS}, guidance={GUIDANCE})')
else:
    print('Stage 1 skipped')


In [ ]:
# Ensure potrace is available (apt-get does NOT persist across Colab reconnects)
import shutil as _shutil, subprocess as _sp
if _shutil.which("potrace") is None:
    print("potrace not found — installing...", end=" ", flush=True)
    _sp.run(["apt-get", "install", "-qq", "-y", "potrace"], check=True)
    print("done")

#@title Stage 1b — Potrace Color Vectorizer
from sklearn.cluster import MiniBatchKMeans

def vectorize_color(image, n_colors=N_COLORS):
    img  = image.resize((512, 512))
    arr  = np.array(img)
    flat = arr.reshape(-1, 3).astype(float)
    km   = MiniBatchKMeans(n_clusters=n_colors, random_state=42, n_init=3, max_iter=100)
    lbls = km.fit_predict(flat)
    cols = km.cluster_centers_.clip(0, 255).astype(int)
    lmap = lbls.reshape(512, 512)
    paths = []
    with tempfile.TemporaryDirectory() as tmp:
        for i, rgb in enumerate(cols):
            mask = ((lmap == i) * 255).astype(np.uint8)
            if mask.sum() < 512: continue
            bmp_p = f'{tmp}/c{i}.bmp'
            svg_p = f'{tmp}/c{i}.svg'
            Image.fromarray(mask, 'L').save(bmp_p)
            r = subprocess.run(
                ['potrace', bmp_p, '--svg', '--turdsize=2', '--alphamax=1', '-o', svg_p],
                capture_output=True
            )
            if r.returncode != 0 or not os.path.exists(svg_p): continue
            with open(svg_p) as f: pt = f.read()
            hc = '#{:02X}{:02X}{:02X}'.format(*rgb)
            for d in re.findall(r'<path[^>]+d="([^"]+)"', pt):
                paths.append(f'  <path d="{d}" fill="{hc}"/>')
    if not paths: return None
    return (
        '<svg viewBox="0 0 512 512" xmlns="http://www.w3.org/2000/svg">\n'
        + '\n'.join(paths) + '\n</svg>'
    )

def rescale_viewbox(svg, size=200):
    return re.sub(r'viewBox="[^"]*"', f'viewBox="0 0 {size} {size}"', svg, count=1)

print('Potrace color vectorizer ready')


In [ ]:
#@title Stage 1c — Select Prompts
TRAIN_PROMPTS = [
    'a red apple', 'a yellow sun', 'a blue circle', 'a green tree', 'a red heart',
    'a yellow star', 'an orange carrot', 'a pink flower', 'a house with red roof',
    'a snowman', 'a rocket', 'a cat face', 'a wifi symbol', 'a battery icon',
    'a music note', 'a play button', 'a gear icon', 'a home icon', 'a mail envelope',
    'a phone icon', 'a camera', 'a lock', 'a mountain', 'a rainbow', 'clouds',
    'a crescent moon', 'a pizza slice', 'a coffee cup', 'an ice cream', 'a cake',
    'a hamburger', 'a donut', 'a watermelon', 'a banana', 'a strawberry',
    'a hot air balloon', 'a treasure chest', 'a lighthouse', 'a bicycle', 'a guitar',
    'circles', 'a spiral', 'squares', 'yin yang', 'a peace sign',
    'a target', 'a smiley', 'thumbs up', 'lightning bolt', 'a car',
]
gen_prompts = []
if Path(RESULTS_FILE).exists():
    with open(RESULTS_FILE) as f: existing = json.load(f)
    records = existing.get('results', existing) if isinstance(existing, dict) else existing
    bad = [r['prompt'] for r in records
           if not r.get('success') or r.get('clip', 99) < 24.0 or r.get('dino', 1.0) < 0.35]
    print(f'Mined {len(bad)} bad prompts from results.json')
    gen_prompts = bad + [p for p in TRAIN_PROMPTS if p not in bad]
else:
    gen_prompts = TRAIN_PROMPTS
gen_prompts = gen_prompts[:NUM_GEN_PROMPTS]
print(f'{len(gen_prompts)} prompts selected')


In [ ]:
#@title Stage 1d — Run Generation Loop  [saves PNGs + SVGs]
if RUN_STAGE1:
    raw_path = Path(DATASET_DIR) / 'raw_dataset.jsonl'
    done = set()
    if raw_path.exists():
        with open(raw_path) as f:
            done = {json.loads(l)['text'] for l in f if l.strip()}
        print(f'Resuming — {len(done)} prompts already done')

    written = failed = 0
    with open(raw_path, 'a', encoding='utf-8') as out_f:
        for i, prompt in enumerate(gen_prompts):
            if prompt in done: continue
            s = slug(prompt)
            print(f'[{i+1:02d}/{len(gen_prompts)}] {prompt}', end=' ... ', flush=True)
            try:
                # Generate PNG
                img = generate_image(prompt, seed=42 + i)
                img.save(f'{S1_IMAGES}/{s}.png')          # <-- save SD PNG

                # Vectorize
                svg = vectorize_color(img, n_colors=N_COLORS)
                if svg is None: print('FAIL (vectorize)'); failed += 1; continue
                svg = rescale_viewbox(svg, 200)
                if not validate_svg(svg): print('FAIL (invalid SVG)'); failed += 1; continue

                # Save SVG file
                with open(f'{S1_SVGS}/{s}.svg', 'w') as svgf:
                    svgf.write(svg)                        # <-- save vectorized SVG

                # Write to dataset
                out_f.write(json.dumps({'text': prompt, 'svg': svg}) + '\n')
                out_f.flush()
                written += 1
                print(f'OK ({count_elements(svg)} elements)')
            except Exception as e:
                print(f'ERROR: {e}'); failed += 1

    print(f'\nStage 1 done — {written} written, {failed} failed')
    print(f'  PNGs : {S1_IMAGES}/')
    print(f'  SVGs : {S1_SVGS}/')
    print(f'  Data : {raw_path}')
    # Free SD 3.5 Medium from RAM before loading Stage-2/3 models
    del sd_pipe, generate_image
    gc.collect(); torch.cuda.empty_cache()
    print("SD 3.5 Medium unloaded — RAM freed for Stage 2")
else:
    print('Stage 1 skipped')


In [ ]:
#@title Stage 1 — Download Outputs
if RUN_STAGE1:
    # Copy jsonl into the outputs folder so it's in the zip
    shutil.copy(str(Path(DATASET_DIR) / 'raw_dataset.jsonl'), f'{S1_SVGS}/../raw_dataset.jsonl')
    zip_folder('./outputs/stage1', './stage1_outputs.zip')
    download_zip('./stage1_outputs.zip', f'({written} PNGs + SVGs)')
else:
    print('Stage 1 skipped — nothing to download')


---
## Stage 2 — VLM Quality Gate
**Qwen2-VL-2B renders each SVG and judges: YES (keep) / NO (discard)**

Saves: `outputs/stage2/kept/*.png`, `outputs/stage2/rejected/*.png`, `dataset/dataset*.jsonl`


In [ ]:
#@title Stage 2 — VLM Quality Gate  [saves kept/rejected renders]
if RUN_STAGE2:
    from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

    print('Loading Qwen2-VL-2B for quality gate...')
    _q = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type='nf4')
    gate_model = Qwen2VLForConditionalGeneration.from_pretrained(
        'Qwen/Qwen2-VL-2B-Instruct', quantization_config=_q, device_map='auto'
    )
    gate_proc = AutoProcessor.from_pretrained('Qwen/Qwen2-VL-2B-Instruct')
    gate_model.eval()
    print('Gate model ready')

    JUDGE_Q = (
        'This is a simple vector illustration. '
        'Does it clearly resemble "{prompt}"? '
        'Reply with exactly one word: YES or NO.'
    )

    def vlm_judge(svg, prompt):
        img = render_svg(svg, 256)
        if img is None: return False
        q = JUDGE_Q.format(prompt=prompt)
        msgs = [{'role': 'user', 'content': [{'type': 'image', 'image': img}, {'type': 'text', 'text': q}]}]
        text = gate_proc.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = gate_proc(text=[text], images=[img], return_tensors='pt').to('cuda')
        with torch.inference_mode():
            out = gate_model.generate(**inputs, max_new_tokens=5, do_sample=False)
        ans = gate_proc.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip().upper()
        return ans.startswith('YES')

    raw_path   = Path(DATASET_DIR) / 'raw_dataset.jsonl'
    clean_path = Path(DATASET_DIR) / 'dataset.jsonl'
    kept = discarded = 0

    with open(raw_path) as rf, open(clean_path, 'w') as wf:
        lines = [l for l in rf if l.strip()]
        for line in tqdm(lines, desc='VLM gate'):
            r    = json.loads(line.strip())
            s    = slug(r['text'])
            keep = vlm_judge(r['svg'], r['text'])

            # Save rendered PNG to kept/ or rejected/
            img = render_svg(r['svg'], 256)
            if img:
                dest = f'{S2_KEPT}/{s}.png' if keep else f'{S2_REJECT}/{s}.png'
                img.save(dest)              # <-- save render

            if keep:
                wf.write(line)
                kept += 1
            else:
                discarded += 1
            print(f'  {"Y" if keep else "X"} {r["text"][:50]}')

    print(f'\nKept: {kept}  Discarded: {discarded}  ({100*discarded/max(kept+discarded,1):.0f}% filtered)')

    # Train / val split (90/10)
    with open(clean_path) as f:
        all_recs = [json.loads(l) for l in f if l.strip()]
    random.seed(42); random.shuffle(all_recs)
    split = max(1, int(len(all_recs) * 0.9))
    for name, subset in [('dataset_train', all_recs[:split]), ('dataset_val', all_recs[split:])]:
        p = Path(DATASET_DIR) / f'{name}.jsonl'
        with open(p, 'w') as f:
            for rec in subset: f.write(json.dumps(rec) + '\n')
        # Copy into outputs folder for download
        shutil.copy(str(p), f'./outputs/stage2/{name}.jsonl')
    shutil.copy(str(clean_path), './outputs/stage2/dataset.jsonl')

    print(f'Split: train={split}  val={len(all_recs)-split}')
    print(f'  Kept renders   : {S2_KEPT}/')
    print(f'  Rejected renders: {S2_REJECT}/')

    del gate_model, gate_proc; gc.collect(); torch.cuda.empty_cache()
else:
    print('Stage 2 skipped')


In [ ]:
#@title Stage 2 — Download Outputs
if RUN_STAGE2:
    zip_folder('./outputs/stage2', './stage2_outputs.zip')
    download_zip('./stage2_outputs.zip', f'(kept={kept}, rejected={discarded})')
else:
    print('Stage 2 skipped — nothing to download')


---
## Stage 3 — QLoRA Fine-Tuning
**Qwen2-VL-2B-Instruct + LoRA on (text, SVG) pairs. Loss only on assistant SVG tokens.**

Saves: `outputs/stage3/training_curve.png`, `outputs/stage3/adapter.zip`


In [ ]:
#@title Stage 3 — QLoRA Fine-Tuning  [saves training curve + adapter zip]
if RUN_STAGE3:
    from transformers import (
        AutoTokenizer, Qwen2VLForConditionalGeneration,
        TrainingArguments, Trainer, BitsAndBytesConfig
    )
    from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

    TRAIN_FILE = str(Path(DATASET_DIR) / 'dataset_train.jsonl')
    VAL_FILE   = str(Path(DATASET_DIR) / 'dataset_val.jsonl')

    print(f'Loading tokenizer: {BASE_MODEL}')
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True, padding_side='right')
    if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

    quant = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True
    )
    print(f'Loading {BASE_MODEL} (4-bit NF4 + double quant)...')
    ft_base = Qwen2VLForConditionalGeneration.from_pretrained(
        BASE_MODEL, quantization_config=quant, device_map='auto', trust_remote_code=True
    )
    ft_base = prepare_model_for_kbit_training(ft_base)

    lora_cfg = LoraConfig(
        task_type=TaskType.CAUSAL_LM, r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.05,
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
        bias='none'
    )
    ft_model = get_peft_model(ft_base, lora_cfg)
    ft_model.print_trainable_parameters()

    def build_chat(prompt, svg):
        return tokenizer.apply_chat_template([
            {'role': 'system',    'content': SVG_SYSTEM_PROMPT},
            {'role': 'user',      'content': f'Generate SVG for: {prompt}'},
            {'role': 'assistant', 'content': svg},
        ], tokenize=False, add_generation_prompt=False)

    def load_jsonl(path):
        with open(path) as f: return [json.loads(l) for l in f if l.strip()]

    class SVGDataset(torch.utils.data.Dataset):
        def __init__(self, records):
            self.records = records
        def __len__(self): return len(self.records)
        def __getitem__(self, idx):
            r   = self.records[idx]
            enc = tokenizer(build_chat(r['text'], r['svg']), truncation=True,
                            max_length=MAX_SEQ, padding=False, return_tensors=None)
            ids  = enc['input_ids']
            lbls = list(ids)
            atoks = tokenizer.encode('<|im_start|>assistant', add_special_tokens=False)
            n, pos = len(atoks), 0
            for i in range(len(ids) - n):
                if ids[i:i+n] == atoks: pos = i + n
            for i in range(pos): lbls[i] = -100
            return {'input_ids': ids, 'labels': lbls, 'attention_mask': enc['attention_mask']}

    class PadCollator:
        def __call__(self, features):
            ml  = max(len(f['input_ids']) for f in features)
            pad = tokenizer.pad_token_id or 0
            ids_b, lbl_b, msk_b = [], [], []
            for f in features:
                pl = ml - len(f['input_ids'])
                ids_b.append(f['input_ids']     + [pad]  * pl)
                lbl_b.append(f['labels']         + [-100] * pl)
                msk_b.append(f['attention_mask'] + [0]    * pl)
            return {
                'input_ids':      torch.tensor(ids_b, dtype=torch.long),
                'labels':         torch.tensor(lbl_b, dtype=torch.long),
                'attention_mask': torch.tensor(msk_b, dtype=torch.long),
            }

    train_ds = SVGDataset(load_jsonl(TRAIN_FILE))
    val_ds   = SVGDataset(load_jsonl(VAL_FILE)) if Path(VAL_FILE).exists() else None
    print(f'Train: {len(train_ds)}  Val: {len(val_ds) if val_ds else 0}')
    if len(train_ds) == 0:
        print("⚠️  No training samples found. "
              "Re-run Stage 1 (potrace must be installed) and Stage 2 first.")
    else:

        train_args = TrainingArguments(
            output_dir=OUTPUT_DIR, num_train_epochs=EPOCHS,
            per_device_train_batch_size=1, gradient_accumulation_steps=8,
            learning_rate=LR, lr_scheduler_type='cosine', warmup_steps=max(1, int(0.05 * max(1, len(train_ds) // 8) * EPOCHS)),
            fp16=True, logging_steps=10,
            eval_strategy='steps' if val_ds else 'no', eval_steps=100 if val_ds else None,
            save_strategy='steps', save_steps=200, save_total_limit=2,
            load_best_model_at_end=bool(val_ds), report_to='none',
            seed=42, remove_unused_columns=False, dataloader_pin_memory=False,
        )
        trainer = Trainer(
            model=ft_model, args=train_args,
            train_dataset=train_ds, eval_dataset=val_ds,
            data_collator=PadCollator(), processing_class=tokenizer,
        )
        print('\nStarting training...')
        trainer.train()

        # Save adapter
        adapter_out = os.path.join(OUTPUT_DIR, 'final_adapter')
        ft_model.save_pretrained(adapter_out)
        tokenizer.save_pretrained(adapter_out)
        ADAPTER_PATH = adapter_out

        # Plot and save training curve
        log_history = trainer.state.log_history
        train_losses = [(e['step'], e['loss']) for e in log_history if 'loss' in e]
        eval_losses  = [(e['step'], e['eval_loss']) for e in log_history if 'eval_loss' in e]
        if train_losses:
            steps_t, losses_t = zip(*train_losses)
            plt.figure(figsize=(8, 4))
            plt.plot(steps_t, losses_t, label='train loss', color='steelblue')
            if eval_losses:
                steps_e, losses_e = zip(*eval_losses)
                plt.plot(steps_e, losses_e, label='val loss', color='orange', ls='--')
            plt.xlabel('Step'); plt.ylabel('Loss')
            plt.title(f'Training Curve — {BASE_MODEL} LoRA r={LORA_R} epochs={EPOCHS}')
            plt.legend(); plt.tight_layout()
            curve_path = f'{S3_OUT}/training_curve.png'
            plt.savefig(curve_path, dpi=150)
            plt.show()
            print(f'  Training curve saved: {curve_path}')

        # Save adapter as zip
        adapter_zip = f'{S3_OUT}/adapter.zip'
        with zipfile.ZipFile(adapter_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
            for f in sorted(Path(adapter_out).rglob('*')):
                if f.is_file(): zf.write(f, f.relative_to(Path(adapter_out).parent))
        size_mb = Path(adapter_zip).stat().st_size / 1e6
        print(f'  Adapter saved: {adapter_zip} ({size_mb:.1f} MB)')

        # Save training log JSON
        with open(f'{S3_OUT}/training_log.json', 'w') as f:
            json.dump(log_history, f, indent=2)

        print(f'\nStage 3 done — adapter: {adapter_out}')
        del ft_model, ft_base, trainer; gc.collect(); torch.cuda.empty_cache()
    else:
        print('Stage 3 skipped')


In [ ]:
#@title Stage 3 — Download Outputs
if RUN_STAGE3:
    # The adapter zip is already in S3_OUT; zip the whole stage3 folder
    zip_folder('./outputs/stage3', './stage3_outputs.zip')
    download_zip('./stage3_outputs.zip', '(training_curve.png + adapter.zip + training_log.json)')
else:
    print('Stage 3 skipped — nothing to download')


---
## Stage 4 — Inference + Iterative Code Correction
**Text → fine-tuned Qwen2-VL → SVG → render → if invalid, correct (up to 3 rounds)**

Saves: `outputs/stage4/svgs/*.svg`, `outputs/stage4/renders/*.png`, `outputs/stage4/stage4_results.json`


In [ ]:
#@title Stage 4a — Load Fine-Tuned Model
if RUN_STAGE4:
    from transformers import Qwen2VLForConditionalGeneration, AutoTokenizer, BitsAndBytesConfig
    from peft import PeftModel

    if not Path(ADAPTER_PATH).exists():
        print(f'Adapter not found at {ADAPTER_PATH}.')
        print('  On Kaggle: add adapter zip as a Dataset and set ADAPTER_PATH in Config.')
    print(f'Loading {BASE_MODEL} + LoRA from {ADAPTER_PATH}...')
    _quant = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type='nf4'
    )
    _base = Qwen2VLForConditionalGeneration.from_pretrained(
        BASE_MODEL, quantization_config=_quant, device_map='auto', trust_remote_code=True
    )
    inf_model = PeftModel.from_pretrained(_base, ADAPTER_PATH)
    inf_model.eval()
    inf_tok = AutoTokenizer.from_pretrained(ADAPTER_PATH, trust_remote_code=True)
    print('Inference model ready')
else:
    print('Stage 4 skipped')


In [ ]:
#@title Stage 4b — Generation + Correction Loop  [saves SVGs + renders]
if RUN_STAGE4:

    def _generate(system_content, user_content):
        chat = inf_tok.apply_chat_template(
            [{'role': 'system', 'content': system_content},
             {'role': 'user',   'content': user_content}],
            tokenize=False, add_generation_prompt=True
        )
        ids     = inf_tok(chat, return_tensors='pt').to(inf_model.device)
        stop_id = inf_tok.encode('</svg>', add_special_tokens=False)[-1]
        with torch.inference_mode():
            out = inf_model.generate(
                **ids, max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False, repetition_penalty=1.05, eos_token_id=stop_id
            )
        return inf_tok.decode(out[0][ids.input_ids.shape[1]:], skip_special_tokens=True).strip()

    CORR_PROMPT = (
        'The following SVG does not render correctly. '
        'Fix it to clearly represent "{prompt}".\n'
        'Current SVG:\n{svg}\n\n'
        'Output ONLY the corrected SVG code.'
    )

    def gen_svg(prompt, max_rounds=MAX_CORR_ROUNDS):
        raw = _generate(SVG_SYSTEM_PROMPT, f'Generate SVG for: {prompt}')
        svg = repair_svg(extract_svg(raw) or raw)
        for rnd in range(max_rounds + 1):
            if validate_svg(svg):
                return {'svg': svg, 'rounds': rnd, 'success': True}
            if rnd == max_rounds: break
            corr = _generate(
                SVG_SYSTEM_PROMPT,
                CORR_PROMPT.format(prompt=prompt, svg=svg)
            )
            svg = repair_svg(extract_svg(corr) or corr)
        return {'svg': svg, 'rounds': max_rounds, 'success': False}

    # Run on test prompts and save all outputs
    test_prompts = [
        'a red apple', 'a yellow star', 'a cat face', 'a rocket',
        'a bicycle', 'a lighthouse', 'a pizza slice', 'a coffee cup',
    ]
    stage4_results = []
    print(f'Generating SVGs for {len(test_prompts)} prompts...\n')

    for prompt in test_prompts:
        s  = slug(prompt)
        t0 = time.time()
        r  = gen_svg(prompt)
        elapsed = time.time() - t0

        # Save SVG file
        with open(f'{S4_SVGS}/{s}.svg', 'w') as svgf:
            svgf.write(r['svg'])                           # <-- save SVG

        # Save rendered PNG
        img = render_svg(r['svg'], 512)
        if img:
            img.save(f'{S4_RENDERS}/{s}.png')              # <-- save render

        status = 'OK' if r['success'] else 'FAIL'
        print(f'  {prompt:<28} {status}  rounds={r["rounds"]}  elems={count_elements(r["svg"])}  ({elapsed:.1f}s)')

        stage4_results.append({
            'prompt':   prompt,
            'success':  r['success'],
            'rounds':   r['rounds'],
            'elements': count_elements(r['svg']),
            'time':     round(elapsed, 1),
            'svg':      r['svg'],
        })

    # Save stage4 results JSON
    with open(f'{S4_SVGS}/../stage4_results.json', 'w') as f:
        json.dump(stage4_results, f, indent=2)

    ok = sum(1 for r in stage4_results if r['success'])
    print(f'\nStage 4: {ok}/{len(test_prompts)} valid')
    print(f'  SVGs    : {S4_SVGS}/')
    print(f'  Renders : {S4_RENDERS}/')

    # Gallery
    valid4 = [r for r in stage4_results if r['success']][:6]
    if valid4:
        fig, axes = plt.subplots(1, len(valid4), figsize=(3*len(valid4), 3))
        if len(valid4) == 1: axes = [axes]
        for ax, r in zip(axes, valid4):
            img = render_svg(r['svg'])
            if img: ax.imshow(img)
            ax.set_title(r['prompt'][:18], fontsize=8); ax.axis('off')
        plt.suptitle('Stage 4 — Fine-tuned Qwen2-VL Outputs')
        plt.tight_layout()
        plt.savefig(f'{S4_RENDERS}/../gallery.png', dpi=150)
        plt.show()
else:
    print('Stage 4 skipped')


In [ ]:
#@title Stage 4 — Download Outputs
if RUN_STAGE4:
    zip_folder('./outputs/stage4', './stage4_outputs.zip')
    download_zip('./stage4_outputs.zip', f'(SVGs + renders + results JSON)')
else:
    print('Stage 4 skipped — nothing to download')


---
## Stage 5 — Evaluation (CLIP + Model Comparison)
**Held-out eval set → CLIP ViT-B/32 scoring → multi-model comparison table**

Saves: `outputs/stage5/svgs/*.svg`, `outputs/stage5/renders/*.png`, `results.json`, plots


In [ ]:
#@title Stage 5a — Load CLIP
if RUN_STAGE5:
    import open_clip
    print('Loading CLIP ViT-B/32...')
    clip_model, _, clip_prep = open_clip.create_model_and_transforms('ViT-B-32', pretrained='openai')
    clip_model = clip_model.to(DEVICE).eval()
    clip_tok   = open_clip.get_tokenizer('ViT-B-32')

    @torch.no_grad()
    def clip_score(img, text):
        img_t = clip_prep(img).unsqueeze(0).to(DEVICE)
        txt_t = clip_tok([text]).to(DEVICE)
        img_f = F.normalize(clip_model.encode_image(img_t), dim=-1)
        txt_f = F.normalize(clip_model.encode_text(txt_t),  dim=-1)
        return (img_f @ txt_f.T).item() * 100

    print('CLIP ready')
else:
    print('Stage 5 skipped')


In [ ]:
#@title Stage 5b — Run Evaluation  [saves SVGs + renders per prompt]
if RUN_STAGE5:
    if Path(EVAL_PROMPTS).exists():
        with open(EVAL_PROMPTS) as f: ep = json.load(f)
        eval_list = ep['prompts']
        print(f'Loaded {len(eval_list)} eval prompts from {EVAL_PROMPTS}')
    else:
        eval_list = [{'id': i, 'prompt': p, 'category': 'misc'} for i, p in enumerate(TRAIN_PROMPTS[:20])]
        print(f'eval_prompts.json not found — using {len(eval_list)} fallback prompts')

    eval_results = []
    print(f'Evaluating {len(eval_list)} prompts...\n')

    for item in tqdm(eval_list):
        prompt   = item['prompt']
        category = item.get('category', '')
        pid      = item.get('id', 0)
        s        = slug(prompt)
        r = {'id': pid, 'prompt': prompt, 'category': category,
             'success': False, 'clip': 0.0, 'elements': 0, 'time': 0.0,
             'svg': '', 'error': None}
        try:
            t0     = time.time()
            result = gen_svg(prompt)
            r['time'] = round(time.time() - t0, 1)

            # Save SVG regardless of validity
            with open(f'{S5_SVGS}/{s}.svg', 'w') as svgf:
                svgf.write(result['svg'])                  # <-- save SVG

            if result['success']:
                img = render_svg(result['svg'], 512)
                if img:
                    img.save(f'{S5_RENDERS}/{s}.png')      # <-- save render
                    r['success']  = True
                    r['svg']      = result['svg']
                    r['clip']     = round(clip_score(img, prompt), 4)
                    r['elements'] = count_elements(result['svg'])
                else:
                    r['error'] = 'render failed'
            else:
                r['error'] = 'SVG invalid after corrections'
        except Exception as e:
            r['error'] = str(e)

        eval_results.append(r)
        status = f'clip={r["clip"]:.1f}' if r['success'] else f'FAIL({r["error"]})'
        print(f'  [{pid:02d}] {prompt[:38]:<38} {status}')

    valid = [r for r in eval_results if r['success']]
    clips = [r['clip'] for r in valid]
    print(f'\n{"="*55}')
    print(f'  Valid : {len(valid)}/{len(eval_results)} ({100*len(valid)/max(len(eval_results),1):.1f}%)')
    if clips:
        print(f'  CLIP  : {np.mean(clips):.2f} +/- {np.std(clips):.2f}')
        print(f'  Range : {min(clips):.2f} - {max(clips):.2f}')
    print(f'{"="*55}')
    print(f'  SVGs    : {S5_SVGS}/')
    print(f'  Renders : {S5_RENDERS}/')
else:
    print('Stage 5 skipped')


In [ ]:
#@title Stage 5c — API Baselines (optional)
baseline_results = {}

if RUN_STAGE5 and eval_results:
    def gen_api_svg(prompt, model_id, api_key):
        USR = f'Generate an SVG illustration for: {prompt}'
        try:
            if 'gpt' in model_id:
                from openai import OpenAI
                c   = OpenAI(api_key=api_key)
                raw = c.chat.completions.create(
                    model=model_id, max_tokens=1500, temperature=0,
                    messages=[{'role': 'system', 'content': SVG_SYSTEM_PROMPT},
                               {'role': 'user',   'content': USR}]
                ).choices[0].message.content or ''
            else:
                from anthropic import Anthropic
                c   = Anthropic(api_key=api_key)
                raw = c.messages.create(
                    model=model_id, max_tokens=1500, system=SVG_SYSTEM_PROMPT,
                    messages=[{'role': 'user', 'content': USR}]
                ).content[0].text
            svg = extract_svg(raw)
            return repair_svg(svg) if svg else None
        except: return None

    api_models = []
    if OPENAI_API_KEY:    api_models.append(('gpt4o_mini',   'gpt-4o-mini',                  OPENAI_API_KEY))
    if ANTHROPIC_API_KEY: api_models.append(('claude_haiku', 'claude-haiku-4-5-20251001', ANTHROPIC_API_KEY))

    for key, model_id, api_key in api_models:
        bl_svgs_dir    = f'./outputs/stage5/baselines/{key}/svgs'
        bl_renders_dir = f'./outputs/stage5/baselines/{key}/renders'
        os.makedirs(bl_svgs_dir,    exist_ok=True)
        os.makedirs(bl_renders_dir, exist_ok=True)

        print(f'\nRunning {key}...')
        res_m = []
        for item in tqdm(eval_list):
            prompt = item['prompt']
            s      = slug(prompt)
            svg    = gen_api_svg(prompt, model_id, api_key)
            ok     = validate_svg(svg) if svg else False
            score  = 0.0
            if svg:
                with open(f'{bl_svgs_dir}/{s}.svg', 'w') as svgf: svgf.write(svg)
            if ok:
                img = render_svg(svg, 512)
                if img:
                    img.save(f'{bl_renders_dir}/{s}.png')
                    score = round(clip_score(img, prompt), 4)
            res_m.append({'prompt': prompt, 'category': item.get('category',''),
                          'success': ok, 'clip': score,
                          'elements': count_elements(svg or ''), 'svg': svg or ''})
            time.sleep(0.3)
        baseline_results[key] = res_m
        vc = [r for r in res_m if r['success']]
        cl = [r['clip'] for r in vc]
        print(f'  {key}: valid={len(vc)}/{len(res_m)}  CLIP={np.mean(cl):.2f}' if cl else f'  {key}: 0 valid')

    if not api_models:
        print('No API keys set — skipping baselines')
        print('Set OPENAI_API_KEY / ANTHROPIC_API_KEY in Config to enable')


In [ ]:
#@title Stage 5d — Comparison Table + Plots
if RUN_STAGE5 and eval_results:
    all_models = {'qwen_lora': eval_results}
    all_models.update(baseline_results)

    # Comparison table
    print(f'\n{"="*70}')
    print('  SVG GENERATION MODEL COMPARISON')
    print(f'{"="*70}')
    print(f'{"Model":<18} {"Valid%":>8} {"CLIP mean":>10} {"CLIP std":>10} {"Avg elems":>10}')
    print(f'{"-"*70}')
    for mname, res in all_models.items():
        vr = [r for r in res if r.get('success')]
        cr = [r['clip'] for r in vr if r.get('clip', 0) > 0]
        er = [r.get('elements', 0) for r in vr]
        pct = 100 * len(vr) / max(len(res), 1)
        print(
            f'{mname:<18} {pct:>7.1f}%'
            f' {np.mean(cr) if cr else float("nan"):>10.2f}'
            f' {np.std(cr)  if cr else float("nan"):>10.2f}'
            f' {np.mean(er) if er else float("nan"):>10.1f}'
        )
    print(f'{"="*70}\n')

    # Per-category breakdown
    if Path(EVAL_PROMPTS).exists():
        with open(EVAL_PROMPTS) as f: ep = json.load(f)
        cats_map = {p['prompt']: p.get('category', '?') for p in ep['prompts']}
        categories = sorted(set(cats_map.values()))
        col = 12
        print(f'{"Category":<14}' + ''.join(f'{m:>{col}}' for m in all_models))
        print('-' * (14 + col * len(all_models)))
        for cat in categories:
            row = f'{cat:<14}'
            for res in all_models.values():
                cv = [r['clip'] for r in res if r.get('success') and cats_map.get(r['prompt']) == cat]
                row += f'{np.mean(cv) if cv else float("nan"):>{col}.2f}'
            print(row)
        print()

    # Plots
    valid_e = [r for r in eval_results if r['success']]
    if len(valid_e) >= 3:
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        clips_e = [r['clip'] for r in valid_e]
        axes[0].hist(clips_e, bins=min(15, len(clips_e)), color='steelblue', edgecolor='black')
        axes[0].axvline(np.mean(clips_e), color='red', ls='--', label=f'mean={np.mean(clips_e):.1f}')
        axes[0].set_xlabel('CLIP Score'); axes[0].set_title('CLIP Distribution (qwen_lora)'); axes[0].legend()

        mnames = list(all_models.keys())
        mclips = [np.mean([r['clip'] for r in all_models[mn] if r.get('success') and r.get('clip',0)>0] or [0])
                  for mn in mnames]
        axes[1].bar(mnames, mclips, color=['#4C72B0','#DD8452','#55A868','#C44E52'][:len(mnames)], edgecolor='black')
        axes[1].set_ylabel('CLIP Mean'); axes[1].set_title('Model Comparison')
        axes[1].set_xticklabels(mnames, rotation=20, ha='right')

        if Path(EVAL_PROMPTS).exists():
            cat_clips = {}
            for r in valid_e:
                c = cats_map.get(r['prompt'], '?')
                cat_clips.setdefault(c, []).append(r['clip'])
            cnames = sorted(cat_clips)
            axes[2].bar(cnames, [np.mean(cat_clips[c]) for c in cnames], color='seagreen', edgecolor='black')
            axes[2].set_xticklabels(cnames, rotation=30, ha='right')
            axes[2].set_ylabel('CLIP Mean'); axes[2].set_title('CLIP by Category')

        plt.tight_layout()
        plt.savefig(f'{S5_RENDERS}/../eval_plots.png', dpi=150)
        plt.show()

    # Gallery — top 6 by CLIP
    top6 = sorted(valid_e, key=lambda r: r['clip'], reverse=True)[:6]
    if top6:
        fig, axes = plt.subplots(1, len(top6), figsize=(3*len(top6), 3))
        if len(top6) == 1: axes = [axes]
        for ax, r in zip(axes, top6):
            img = render_svg(r['svg'])
            if img: ax.imshow(img)
            ax.set_title(f'{r["prompt"][:16]}\nCLIP:{r["clip"]:.1f}', fontsize=7); ax.axis('off')
        plt.suptitle('Top 6 Results by CLIP Score', fontsize=11)
        plt.tight_layout()
        plt.savefig(f'{S5_RENDERS}/../gallery.png', dpi=150)
        plt.show()


In [ ]:
#@title Stage 5e — Save results.json + Download Stage 5
if RUN_STAGE5 and eval_results:
    # Save results.json and results.csv
    summary = save_results(eval_results, RESULTS_FILE)
    shutil.copy(RESULTS_FILE,                  f'{S5_RENDERS}/../results.json')
    shutil.copy(RESULTS_FILE.replace('.json','.csv'), f'{S5_RENDERS}/../results.csv')
    print(f'results.json  valid={summary["n_valid"]}/{summary["n_total"]}  CLIP={summary["clip_mean"]:.2f}')

    # Save baseline results
    for mname, res in baseline_results.items():
        path = f'./outputs/stage5/baselines/{mname}/results.json'
        with open(path, 'w') as f: json.dump({'model': mname, 'results': res}, f, indent=2)

    # Download stage5 zip
    zip_folder('./outputs/stage5', './stage5_outputs.zip')
    download_zip('./stage5_outputs.zip', '(SVGs + renders + results + plots)')
else:
    print('Stage 5 skipped or no results')


In [ ]:
#@title Download Everything (all stages in one zip)
print('Zipping all outputs...')
all_zip = 'diffusvg_all_outputs.zip'
with zipfile.ZipFile(all_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(Path('./outputs').rglob('*')):
        if f.is_file():
            zf.write(f, f.relative_to('.'))
    # Also include the top-level JSONLs and results
    for extra in [RESULTS_FILE,
                  RESULTS_FILE.replace('.json', '.csv'),
                  f'{DATASET_DIR}/raw_dataset.jsonl',
                  f'{DATASET_DIR}/dataset.jsonl',
                  f'{DATASET_DIR}/dataset_train.jsonl',
                  f'{DATASET_DIR}/dataset_val.jsonl']:
        if Path(extra).exists():
            zf.write(extra)

size_mb = Path(all_zip).stat().st_size / 1e6
n_files = sum(1 for f in Path('./outputs').rglob('*') if f.is_file())
print(f'Total: {n_files} files, {size_mb:.1f} MB -> {all_zip}')
print('\nContents:')
print('  outputs/stage1/images/   - SD-generated PNGs')
print('  outputs/stage1/svgs/     - vectorized SVGs')
print('  outputs/stage2/kept/     - VLM-approved renders')
print('  outputs/stage2/rejected/ - VLM-rejected renders')
print('  outputs/stage3/          - training curve + adapter.zip')
print('  outputs/stage4/svgs/     - fine-tuned model SVGs')
print('  outputs/stage4/renders/  - rendered PNGs')
print('  outputs/stage5/svgs/     - eval SVGs')
print('  outputs/stage5/renders/  - eval rendered PNGs')
print('  outputs/stage5/          - results.json, plots, gallery')

download_zip(all_zip)
